In [2]:
import os
import torch
from transformers import AutoModel

In [3]:
MODEL_PATH = "./omnivoice_qwen3"
OUTPUT_DIR = "onnx_qwen3"
ONNX_FILE = os.path.join(OUTPUT_DIR, "model.onnx")

In [5]:
BATCH = 1                 
BATCH_X2 = BATCH * 2      
SEQ_LEN = 220
HIDDEN_SIZE = 1024

base_model = AutoModel.from_pretrained(
    MODEL_PATH,
    dtype=torch.float32,
    trust_remote_code=True,
)
base_model.eval()
base_model.to("cuda")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3Model(
  (embed_tokens): Embedding(151676, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [6]:
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, inputs_embeds, attention_mask):
        outputs = self.model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            return_dict=True,
        )
        return outputs[0]

model = ModelWrapper(base_model)

inputs_embeds = torch.randn(BATCH_X2, SEQ_LEN, HIDDEN_SIZE, dtype=torch.float32).to("cuda")
attention_mask = torch.ones(BATCH, 1, SEQ_LEN, SEQ_LEN, dtype=torch.float32).to("cuda")

In [7]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
torch.onnx.export(
    model,
    args=(inputs_embeds, attention_mask),
    f=ONNX_FILE,
    input_names=["inputs_embeds", "attention_mask"],
    output_names=["last_hidden_state"],
    dynamic_axes={
        "inputs_embeds": {
            0: "batch_x2",
            1: "sequence_length",
        },
        "attention_mask": {
            0: "batch",
            2: "sequence_length",
            3: "sequence_length",
        },
        "last_hidden_state": {
            0: "batch_x2",
            1: "sequence_length",
        },
    },
    opset_version=17,
    verbose=False,
    external_data=False,
)

/tmp/ipykernel_55152/2434161527.py:1: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
/tmp/ipykernel_55152/2434161527.py:1: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1101 21:10:10.857000 55152 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider settin

ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.14.1+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"inputs_embeds"<FLOAT,[batch_x2,sequence_length,1024]>,
                %"attention_mask"<FLOAT,[batch,1,sequence_length,sequence_length]>
            ),
            outputs=(
                %"last_hidden_state"<FLOAT,[batch_x2,sequence_length,1024]>
            ),
            initializers=(
                %"model.layers.0.self_attn.q_norm.weight"<FLOAT,[128]>{TorchTensor(...)},
                %"model.layers.0.self_attn.k_norm.weight"<FLOAT,[128]>{TorchTensor(...)},
                %"model.layers.1.self_attn.q_norm.weight"<FLOAT,[128]>{TorchTensor(...)},
                %"model.layers.1.self_attn.k_norm.weight"<FLOAT,[128]>{TorchTensor(...)},
                %"

In [1]:
import os
import onnx
from pathlib import Path

In [4]:
m = onnx.load(ONNX_FILE) 
onnx.save_model(m, os.path.join(OUTPUT_DIR, "model_single.onnx"),
                save_as_external_data=False)

In [5]:
model_dir = Path("onnx_qwen3")

(model_dir / "model.onnx").unlink(missing_ok=True)
(model_dir / "model.onnx.data").unlink(missing_ok=True)
(model_dir / "model_single.onnx").rename(model_dir / "model.onnx")

PosixPath('onnx_qwen3/model.onnx')